# ccdp — train the VMMRdb make/model/year identifier on Kaggle

Continue-trains the 196-class Stanford identifier the **full VMMRdb (~9,170 classes)** — which flows straight into
Variant D / multi-car costing (more cars named → better make/segment cost).

### Before you run — notebook settings (right sidebar)
1. **Accelerator** → GPU (T4 ×2 or P100).
2. **Internet** → On (to clone the repo + fetch the base weights).
3. **Add Input** → search **`vmmrdb-dataset`** (prabashwara) and add it — it mounts read-only under `/kaggle/input/` (no download).

Full dataset (~9,170 classes, ~291k images), ~12 epochs ≈ **6–10 h** on a T4 — fits a 12 h session; the 30 h/week quota covers re-runs.


## 1. GPU check


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'enable GPU in settings')

## 2. Clone repo + install

**⚠️ Turn Internet ON first** (Settings → Internet) — Kaggle blocks network by default, so without it
`git clone` fails with *Could not resolve host*. Enabling it needs a phone-verified Kaggle account (free).


In [ ]:
%cd /kaggle/working
# NOTE: requires Internet = ON (Settings panel). Kaggle needs a phone-verified account.
# Private repo? use a token:  https://<TOKEN>@github.com/theDocWho/...
!rm -rf car-crash-fix-amount-predictor
!git clone --depth 1 https://github.com/theDocWho/car-crash-fix-amount-predictor.git
%cd /kaggle/working/car-crash-fix-amount-predictor
!pip -q install -e '.[ml]'

## 3. SSL cert (for fetching the base weights)


In [ ]:
import os, certifi
os.environ['SSL_CERT_FILE']=certifi.where()
os.environ['REQUESTS_CA_BUNDLE']=certifi.where()
print('ok')

## 4. Point the loader at the mounted VMMRdb dataset (no download)

Symlinks the Kaggle input into the path the loader expects; auto-detects the dataset folder by name.


In [ ]:
import os, glob
cands = [d for d in glob.glob('/kaggle/input/*') if 'vmmr' in d.lower()] or glob.glob('/kaggle/input/*')
assert cands, 'Add the vmmrdb-dataset input via Add Data (right sidebar).'
src = cands[0]; print('VMMRdb input:', src)
os.makedirs('data/raw', exist_ok=True)
os.system(f"ln -sfn '{src}' data/raw/vmmrdb-dataset")
# sanity: count class folders (dirs with images)
from ccdp.data import vmmrdb
counts = vmmrdb._class_dir_counts('data/raw/vmmrdb-dataset')
print(f'class folders found: {len(counts)}  total images: {sum(counts.values())}')

## 5. Fetch the base identifier (196-class) to warm-start from


In [ ]:
!ccdp costing init || true
!mkdir -p checkpoints/production
![ -f checkpoints/production/identifier.pt] || curl -fL --retry 3 -o checkpoints/production/identifier.pt \
   https://github.com/theDocWho/car-crash-fix-amount-predictor/releases/download/v0.2.0/identifier.pt

## 6. Continue-train on the full VMMRdb (~9,170 classes)

Warm-starts the 196-class ResNet-50, swaps the head to ~9,170 classes, two-stage fine-tunes. `--top-n 0` = full dataset; add `--epochs-stage2 6`
for a shorter run.


In [ ]:
!ccdp train identifier-continue --dataset vmmrdb --top-n 0 --batch-size 64 --num-workers 4

## 7. Promote + export the trained identifier


In [ ]:
!ccdp registry list --variant identifier | tail -5
# promote the run you just trained (replace <run_id> with the id above):
# !ccdp registry promote <run_id> identifier
import shutil, glob, os
src = sorted(glob.glob('checkpoints/identifier/run_*identifier_vmmrdb/best.pt'))
if src:
    shutil.copy(os.path.realpath(src[-1]), '/kaggle/working/identifier.pt')
    print('exported -> /kaggle/working/identifier.pt  (download from the Output tab)')
else:
    print('no best.pt yet — check the training output above')

## 8. Deploy

Download `identifier.pt` from the **Output** tab (or save the notebook output), then attach it to the GitHub release
the app pulls from:
```bash
gh release upload v0.2.0 identifier.pt --clobber
```
On the next Space boot the new identifier is fetched and recognises the full VMMRdb make/model set — Variant D and multi-car mode
benefit automatically (more cars named → better make/segment cost). The model self-describes its classes (names
embedded in the checkpoint), so no extra mapping file is needed.
